# 05 — Validation against Manual Measurements

Purpose: Compare estimated fit signals with ground-truth manual measurements.

Metrics:
- Per-dimension MAE (cm)
- Size recommendation accuracy (recommendation vs. the size that actually fit)
- Confidence calibration (does high confidence correlate with low error?)

Inputs:
- data/processed/measurements/fit_signals.csv
- data/processed/measurements/recommendations.csv
- data/garment/manual_measurements.csv (ground truth, prepared separately)

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

SIGNALS_CSV = ROOT / "data" / "processed" / "measurements" / "fit_signals.csv"
REC_CSV     = ROOT / "data" / "processed" / "measurements" / "recommendations.csv"
MANUAL_CSV  = ROOT / "data" / "garment" / "manual_measurements.csv"

In [ ]:
sig = pd.read_csv(SIGNALS_CSV)
man = pd.read_csv(MANUAL_CSV)
df = sig.merge(man, on="user_id", suffixes=("_est", "_manual"))
df.head()

In [ ]:
# Per-dimension MAE
pairs = [
    ("shoulder_width_cm_est", "shoulder_width_cm_manual"),
    ("hip_width_cm_est", "hip_width_cm_manual"),
    ("torso_length_cm_est", "torso_length_cm_manual"),
    ("leg_length_cm_est", "leg_length_cm_manual"),
    ("arm_length_cm_est", "arm_length_cm_manual"),
]
summary = []
for est, gt in pairs:
    if est in df and gt in df:
        err = (df[est] - df[gt]).abs()
        summary.append({"dim": est.replace("_est", ""), "mae_cm": err.mean(), "n": err.notna().sum()})
pd.DataFrame(summary)

In [ ]:
# Size recommendation accuracy
rec = pd.read_csv(REC_CSV)
# Assumes manual_measurements.csv has an actual_good_size column
if "actual_good_size" in man.columns:
    merged = rec.merge(man[["user_id", "actual_good_size"]], on="user_id")
    acc = (merged["size"] == merged["actual_good_size"]).mean()
    print(f"size recommendation accuracy: {acc:.1%} (n={len(merged)})")
else:
    print("manual_measurements.csv has no actual_good_size column.")